<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/04_clasificacion_mobilenetv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación con MobileNetV2

Tarea 2 de la consigna: clasificar cada imagen en **con paneles / sin paneles** con MobileNetV2 de Keras.

Lo único que se elige es `PESOS`: el notebook corre, uno detrás de otro, todos los experimentos de `EXPERIMENTOS`
con esos pesos.

| `PESOS` | Experimentos que corre |
|---|---|
| `"imagenet"` | `mobilenetv2_imagenet_congelado`: extractor congelado, solo se entrena la cabeza<br>`mobilenetv2_imagenet_fine_tuning`: cabeza y después la última parte del extractor<br>`mobilenetv2_imagenet_ft_completo`: cabeza y después todo el extractor |
| `None` | `mobilenetv2_aleatorio`: sin pre-entrenamiento, toda la red desde pesos aleatorios |

1. Poné tu nombre en `INTEGRANTE` y corré el setup (GPU T4: *Entorno de ejecución → Cambiar tipo de entorno*).
2. Elegí `PESOS` y corré todo (*Entorno de ejecución → Ejecutar todas*).
3. Cada experimento se guarda con el formato común del equipo en `Mi unidad/PDI-TP/resultados/mobilenetv2/<modo>/`:
   pesos (`modelo.keras` y checkpoints por etapa), `metricas.json`, `predicciones_val.csv` (predicción contra el
   real) y `config.json`. Si la sesión se corta, volver a correr todo: los modos que ya están completos se saltean.
4. Para guardar también los gráficos: *Archivo → Guardar una copia en Drive*.

In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

INTEGRANTE = os.environ.get("PDI_INTEGRANTE", "nombre")  # reemplazar por tu nombre

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-training.txt")], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

from pdi_tp import entorno, datos, resultados, split
rutas = entorno.preparar(INTEGRANTE)


## Configuración

Cada experimento pisa los valores de `BASE` que cambian. Para una variante nueva, agregar una entrada a
`EXPERIMENTOS` con un nombre único (y hacer push, así queda registrada para el equipo).

- `modo="congelado"`: `base.trainable = False`, se entrena solo la cabeza.
- `modo="fine_tuning"`: etapa 1 igual que `congelado`; etapa 2 descongela el extractor desde la capa
  `descongelar_desde` (0 = completo; MobileNetV2 tiene 154 capas) y recompila con un learning rate bajo.
- `modo="completo"`: toda la red entrenable desde el inicio. Es el único modo con sentido para `pesos=None`.

`img_size` es el tamaño con el que entra la imagen a la red (las originales son de 256 px). Keras tiene pesos de
ImageNet para 96, 128, 160, 192 y 224; con otro tamaño avisa y usa los de 224, que también funcionan.

In [ ]:
PESOS = "imagenet"  # <- lo único a cambiar: "imagenet" o None (pesos aleatorios)
REPETIR = False     # True vuelve a entrenar también los experimentos que ya tienen resultados guardados

MODELO = "mobilenetv2"  # carpeta en Drive: resultados/mobilenetv2/<nombre del experimento>/

EXPERIMENTOS = {
    "imagenet_congelado": {"pesos": "imagenet", "modo": "congelado"},
    "imagenet_fine_tuning": {"pesos": "imagenet", "modo": "fine_tuning", "descongelar_desde": 100},
    "imagenet_ft_completo": {"pesos": "imagenet", "modo": "fine_tuning", "descongelar_desde": 0},
    "aleatorio": {"pesos": None, "modo": "completo"},
}

BASE = {
    "modelo": "MobileNetV2",
    "pesos": "imagenet",         # "imagenet" o None (pesos aleatorios)
    "modo": "fine_tuning",       # "congelado", "fine_tuning" o "completo"
    "img_size": 224,
    "batch_size": 32,
    "augmentation": True,
    "dropout": 0.2,
    "class_weight": True,        # compensa el desbalance (hay muchas más imágenes con paneles)
    # Etapa 1 (cabeza, extractor congelado): modos "congelado" y "fine_tuning"
    "epocas_cabeza": 15,
    "lr_cabeza": 1e-3,
    # Etapa 2 (extractor descongelado): modo "fine_tuning"
    "epocas_fine_tuning": 15,
    "lr_fine_tuning": 1e-5,
    "descongelar_desde": 100,    # 0 = todo el extractor
    # Modo "completo": toda la red desde el inicio
    "epocas_completo": 40,
    "lr_completo": 1e-3,
    "paciencia": 6,              # early stopping sobre val_loss
    "umbral": 0.5,
    "semilla": 42,
}

a_correr = {}
for nombre, cambios in EXPERIMENTOS.items():
    config = {**BASE, **cambios}
    assert config["modo"] in ("congelado", "fine_tuning", "completo"), nombre
    if config["pesos"] is None and config["modo"] != "completo":
        raise ValueError(f"{nombre}: con pesos aleatorios congelar el extractor no tiene sentido, usar modo='completo'.")
    if config["pesos"] != PESOS:
        continue
    if not REPETIR and resultados.corrida_completa(resultados.ruta_corrida(rutas, MODELO, nombre)):
        print(f"Ya hecho, se saltea: {nombre}")
        continue
    a_correr[nombre] = config

print("A correr:", list(a_correr) or "nada (todos hechos; REPETIR = True para volver a entrenarlos)")

## Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

archivos = split.cargar_split(rutas)


def cargar(imagenes):
    """Imágenes RGB uint8 (n, 256, 256, 3) y etiquetas 1 = con paneles, 0 = sin paneles."""
    X = np.stack([datos.leer_imagen(p) for p in imagenes])
    y = np.array([datos.tiene_paneles(datos.label_de(rutas, p)) for p in imagenes], dtype="float32")
    return X, y


X_train, y_train = cargar(archivos["train"])
X_val, y_val = cargar(archivos["val"])

pd.DataFrame({
    "imágenes": [len(y_train), len(y_val)],
    "con paneles": [int(y_train.sum()), int(y_val.sum())],
    "sin paneles": [int((1 - y_train).sum()), int((1 - y_val).sum())],
    "% con paneles": [100 * y_train.mean(), 100 * y_val.mean()],
}, index=["train", "val"]).round(1)

El dataset está desbalanceado: la gran mayoría de las imágenes tiene paneles. Un clasificador que siempre diga
"con paneles" ya logra una accuracy alta, así que hay que mirar sobre todo **precision, recall y F1** y la matriz
de confusión. Con `class_weight=True` cada clase pesa en la loss en proporción inversa a su frecuencia.

**Augmentation** (solo en train): rotaciones de 90° y espejados (una foto aérea no tiene "arriba"), y pequeños
cambios de brillo y contraste.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def aumentar(x):
    x = tf.image.rot90(x, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, max_delta=20.0)
    x = tf.image.random_contrast(x, 0.85, 1.15)
    return tf.clip_by_value(x, 0.0, 255.0)


def armar_ds(X, y, entrenamiento, config):
    ds = tf.data.Dataset.from_tensor_slices((X, y)).map(lambda x, y: (tf.cast(x, tf.float32), y))
    if entrenamiento:
        ds = ds.shuffle(len(X), seed=config["semilla"])
        if config["augmentation"]:
            ds = ds.map(lambda x, y: (aumentar(x), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(config["batch_size"]).prefetch(AUTOTUNE)


def pesos_por_clase(y):
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    return {0: len(y) / (2 * n_neg), 1: len(y) / (2 * n_pos)}


print("class_weight:", pesos_por_clase(y_train))

# Un batch aumentado, para chequear que la augmentation tiene sentido
x_ej, y_ej = next(iter(armar_ds(X_train, y_train, entrenamiento=True, config=BASE)))
fig, axs = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, x, y in zip(axs, x_ej, y_ej):
    ax.imshow(x.numpy().astype("uint8"))
    ax.set_title("con paneles" if y else "sin paneles", fontsize=9)
    ax.set_axis_off()
plt.show()

## Modelo

`MobileNetV2(include_top=False)` como extractor, y una cabeza propia: *global average pooling → dropout → dense
sigmoide*. El modelo recibe las imágenes en [0, 255] y adentro las redimensiona a `img_size` y las lleva a
[-1, 1], que es lo que espera MobileNetV2 (igual que `mobilenet_v2.preprocess_input`).

Con pesos de ImageNet la base se llama con `training=False`: así sus capas de *batch normalization* siguen usando
las estadísticas de ImageNet aunque después se descongele el extractor. Si no, en el fine-tuning se actualizarían
con batches chicos y se arruinaría lo aprendido (ver la guía de transfer learning de Keras). Con pesos aleatorios
no hay estadísticas que preservar, así que la base se llama normalmente.

In [ ]:
def armar_modelo(config):
    s = config["img_size"]
    base = keras.applications.MobileNetV2(input_shape=(s, s, 3), include_top=False, weights=config["pesos"])

    entradas = keras.Input(shape=(256, 256, 3))
    x = keras.layers.Resizing(s, s)(entradas) if s != 256 else entradas
    x = keras.layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base(x, training=False) if config["pesos"] == "imagenet" else base(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(config["dropout"])(x)
    salidas = keras.layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(entradas, salidas, name=config["modelo"]), base

## Entrenamiento y evaluación

`correr_experimento` hace todo para un experimento:

1. **Entrena** por etapas. Cada vez que cambia `trainable` hay que volver a compilar, así que cada etapa compila y
   entrena. Se guarda en Drive el mejor modelo de cada etapa según `val_loss`, y el early stopping restaura esos
   pesos al terminar.
2. **Evalúa sobre val** con las métricas que pide la consigna: matriz de confusión, accuracy, precision, recall y
   F1, con la clase positiva = *con paneles*. Muestra también los errores, para entender qué le cuesta al modelo.
3. **Guarda** el modelo, las probabilidades por imagen de val (para comparar después con el detector usado como
   clasificador sobre las mismas imágenes) y el JSON del experimento.

In [ ]:
import time

from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)


def entrenar(nombre, model, etapa, lr, epocas, train_ds, val_ds, config):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")],
    )
    entrenables = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    print(f"--- Etapa {etapa}: lr={lr}, {entrenables:,} parámetros entrenables ---")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=config["paciencia"], restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(str(resultados.ruta_corrida(rutas, MODELO, nombre) / f"checkpoint_{etapa}.keras"),
                                        monitor="val_loss", save_best_only=True),
    ]
    class_weight = pesos_por_clase(y_train) if config["class_weight"] else None
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epocas,
                     class_weight=class_weight, callbacks=callbacks, verbose=2)
    return etapa, hist.history


def graficar_historial(nombre, historiales):
    fig, axs = plt.subplots(1, 3, figsize=(16, 4))
    total = sum(len(h["loss"]) for _, h in historiales)
    for ax, metrica in zip(axs, ["loss", "accuracy", "recall"]):
        inicio = 0
        for etapa, h in historiales:
            epocas = range(inicio + 1, inicio + len(h[metrica]) + 1)
            ax.plot(epocas, h[metrica], label=f"train ({etapa})")
            ax.plot(epocas, h[f"val_{metrica}"], "--", label=f"val ({etapa})")
            inicio += len(h[metrica])
            if inicio < total:
                ax.axvline(inicio + 0.5, color="gray", lw=0.8)
        ax.set_title(metrica)
        ax.set_xlabel("época")
    axs[0].legend(fontsize=8)
    fig.suptitle(nombre)
    plt.tight_layout()
    plt.show()


def evaluar(nombre, model, val_ds, config):
    prob_val = model.predict(val_ds, verbose=0).ravel()
    pred_val = (prob_val >= config["umbral"]).astype(int)
    y_true = y_val.astype(int)

    metricas = {
        "accuracy": accuracy_score(y_true, pred_val),
        "precision": precision_score(y_true, pred_val, zero_division=0),
        "recall": recall_score(y_true, pred_val, zero_division=0),
        "f1": f1_score(y_true, pred_val, zero_division=0),
        "recall_sin_paneles": recall_score(y_true, pred_val, pos_label=0, zero_division=0),
        "f1_macro": f1_score(y_true, pred_val, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred_val),
        "auc": roc_auc_score(y_true, prob_val),
        "accuracy_siempre_con_paneles": float(y_true.mean()),  # referencia: clasificador trivial
        "matriz_confusion": confusion_matrix(y_true, pred_val).tolist(),  # filas = real [sin, con]
    }
    print(classification_report(y_true, pred_val, target_names=["sin paneles", "con paneles"], digits=3,
                                zero_division=0))
    print({k: round(v, 4) for k, v in metricas.items() if isinstance(v, float)})

    ConfusionMatrixDisplay.from_predictions(y_true, pred_val, display_labels=["sin paneles", "con paneles"],
                                            cmap="Blues")
    plt.title(nombre)
    plt.show()

    predicciones = pd.DataFrame({
        "archivo": [p.name for p in archivos["val"]],
        "real": y_true,
        "prob_con_paneles": prob_val,
        "pred": pred_val,
    })
    return metricas, predicciones


def mostrar_errores(predicciones):
    errores = predicciones[predicciones["real"] != predicciones["pred"]]
    print(f"{len(errores)} errores de {len(predicciones)}")
    n = min(len(errores), 16)
    if not n:
        return
    fig, axs = plt.subplots(2, 8, figsize=(18, 5))
    for ax in axs.ravel():
        ax.set_axis_off()
    for ax, (i, fila) in zip(axs.ravel(), errores.head(n).iterrows()):
        datos.dibujar_cajas(ax, rutas, archivos["val"][i])
        real = "con" if fila["real"] else "sin"
        ax.set_title(f"real: {real} | p={fila['prob_con_paneles']:.2f}", fontsize=8)
    plt.tight_layout()
    plt.show()


def correr_experimento(nombre, config):
    carpeta = resultados.carpeta_corrida(rutas, MODELO, nombre, reiniciar=REPETIR)
    keras.backend.clear_session()
    keras.utils.set_random_seed(config["semilla"])
    train_ds = armar_ds(X_train, y_train, entrenamiento=True, config=config)
    val_ds = armar_ds(X_val, y_val, entrenamiento=False, config=config)  # sin mezclar: orden de archivos["val"]
    model, base = armar_modelo(config)

    inicio = time.perf_counter()
    historiales = []
    if config["modo"] in ("congelado", "fine_tuning"):
        base.trainable = False
        historiales.append(entrenar(nombre, model, "cabeza", config["lr_cabeza"], config["epocas_cabeza"],
                                    train_ds, val_ds, config))
    if config["modo"] == "fine_tuning":
        base.trainable = True
        for capa in base.layers[:config["descongelar_desde"]]:
            capa.trainable = False
        historiales.append(entrenar(nombre, model, "fine_tuning", config["lr_fine_tuning"],
                                    config["epocas_fine_tuning"], train_ds, val_ds, config))
    if config["modo"] == "completo":
        base.trainable = True
        historiales.append(entrenar(nombre, model, "completo", config["lr_completo"], config["epocas_completo"],
                                    train_ds, val_ds, config))
    segundos = time.perf_counter() - inicio
    graficar_historial(nombre, historiales)

    metricas, predicciones = evaluar(nombre, model, val_ds, config)
    metricas["epocas_por_etapa"] = {etapa: len(h["loss"]) for etapa, h in historiales}
    metricas["segundos_entrenamiento"] = round(segundos, 1)
    mostrar_errores(predicciones)

    model.save(carpeta / "modelo.keras")
    pd.concat([pd.DataFrame(h).assign(etapa=etapa) for etapa, h in historiales]).to_csv(
        carpeta / "historia.csv", index_label="epoca")
    resultados.guardar_corrida(
        rutas, carpeta, archivos=archivos["val"], probabilidad=predicciones["prob_con_paneles"],
        params=config, umbral=config["umbral"],
        metricas_extra={k: metricas[k] for k in ("epocas_por_etapa", "segundos_entrenamiento")})

## Correr los experimentos

In [ ]:
for nombre, config in a_correr.items():
    print("=" * 110)
    print(nombre, "→", EXPERIMENTOS[nombre])
    print("=" * 110)
    correr_experimento(nombre, config)

## Comparación de modos: ¿cuál es el óptimo?

Se comparan los experimentos de MobileNetV2 guardados en `resultados/mobilenetv2/` (los de ImageNet y el de pesos
aleatorios, aunque se hayan corrido en sesiones distintas o por distintos integrantes) sobre las mismas 162 imágenes
de val.

**Criterio: F1 macro** (promedio de la F1 de cada clase); desempata el AUC. No sirven la accuracy ni la F1 de
*con paneles*: como casi todas las imágenes tienen paneles, el clasificador trivial que siempre dice "con paneles"
saca valores altos en las dos (fila *siempre con paneles*, y línea punteada en los gráficos). La F1 macro le da el
mismo peso a la clase *sin paneles*, así que un modelo que la ignora queda abajo.

Las métricas se recalculan desde la matriz de confusión guardada en `metricas.json`.

In [ ]:
def metricas_desde_matriz(matriz):
    (tn, fp), (fn, tp) = matriz  # filas = real [sin, con], columnas = predicho [sin, con]

    def div(a, b):
        return a / b if b else 0.0

    def f1(p, r):
        return div(2 * p * r, p + r)

    precision_con, recall_con = div(tp, tp + fp), div(tp, tp + fn)
    precision_sin, recall_sin = div(tn, tn + fn), div(tn, tn + fp)
    return {
        "f1_macro": (f1(precision_con, recall_con) + f1(precision_sin, recall_sin)) / 2,
        "balanced_accuracy": (recall_con + recall_sin) / 2,
        "recall_sin_paneles": recall_sin,
        "recall": recall_con,
        "precision": precision_con,
        "f1": f1(precision_con, recall_con),
        "accuracy": div(tp + tn, tp + tn + fp + fn),
    }


tabla = resultados.cargar_corridas(rutas)
propios = tabla[(tabla["modelo"] == MODELO) & tabla["completa"]]

if propios.empty:
    print("Todavía no hay experimentos de MobileNetV2 guardados en", resultados.ruta_corrida(rutas, MODELO, "x").parent)
else:
    comparacion = pd.DataFrame(
        [{**metricas_desde_matriz(f["matriz_confusion"]), "auc": f["roc_auc"],
          "pesos": f["params.pesos"], "modo": f["params.modo"],
          "descongelar_desde": f["params.descongelar_desde"] if f["params.modo"] == "fine_tuning" else None,
          "integrante": f["integrante"]}
         for _, f in propios.iterrows()],
        index=propios["modo"],
    ).sort_values(["f1_macro", "auc"], ascending=False)

    n_pos, n_neg = int(y_val.sum()), int(len(y_val) - y_val.sum())
    trivial = {**metricas_desde_matriz([[0, n_neg], [0, n_pos]]), "auc": 0.5}

    mejor = comparacion.index[0]
    fila = comparacion.iloc[0]
    print(f"Óptimo según F1 macro: {mejor}  (pesos={fila['pesos']}, modo={fila['modo']})")
    print(f"  F1 macro {fila['f1_macro']:.3f} vs {trivial['f1_macro']:.3f} del trivial | "
          f"recall sin paneles {fila['recall_sin_paneles']:.3f} | AUC {fila['auc']:.3f}")
    if len(comparacion) > 1:
        segundo = comparacion.iloc[1]
        print(f"  Diferencia con el segundo ({comparacion.index[1]}): "
              f"{fila['f1_macro'] - segundo['f1_macro']:+.3f} de F1 macro. Con 20 imágenes 'sin paneles' en val, "
              "una sola imagen mueve el recall de esa clase en 0.05: diferencias chicas pueden ser ruido.")

    display(pd.concat([comparacion, pd.DataFrame([trivial], index=["(siempre con paneles)"])]).round(3))

In [ ]:
if not propios.empty:
    GRAFICO = {"f1_macro": "F1 macro (criterio)", "balanced_accuracy": "Balanced accuracy",
               "recall_sin_paneles": "Recall sin paneles", "auc": "AUC"}
    DESTACADO, RESTO, TINTA, TINTA_SEC = "#2a78d6", "#c9c8c2", "#0b0b0b", "#52514e"

    orden = comparacion.index[::-1]  # el mejor arriba
    fig, axs = plt.subplots(1, len(GRAFICO), figsize=(16, 0.55 * len(orden) + 1.4), sharey=True)
    for ax, (columna, titulo) in zip(axs, GRAFICO.items()):
        valores = comparacion.loc[orden, columna]
        ax.barh(range(len(orden)), valores, height=0.6,
                color=[DESTACADO if n == mejor else RESTO for n in orden])
        for y, v in enumerate(valores):
            ax.text(v + 0.015, y, f"{v:.3f}", va="center", fontsize=9, color=TINTA)
        ax.axvline(trivial[columna], color=TINTA_SEC, ls="--", lw=1)
        ax.set_xlim(0, 1.15)
        ax.set_title(titulo, loc="left", fontsize=11, color=TINTA)
        ax.tick_params(colors=TINTA_SEC, labelsize=9)
        ax.tick_params(axis="y", length=0)
        ax.grid(axis="x", color="#e5e4df", lw=0.6)
        ax.set_axisbelow(True)
        for lado in ("top", "right", "left"):
            ax.spines[lado].set_visible(False)
    axs[0].set_yticks(range(len(orden)), orden)
    axs[0].tick_params(axis="y", length=0, labelsize=10, labelcolor=TINTA)
    fig.text(0.01, 0.005, "Línea punteada: clasificador trivial que siempre dice 'con paneles'. "
             "En azul, el óptimo según F1 macro.", fontsize=9, color=TINTA_SEC)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()

Todos los experimentos del equipo:

In [ ]:
tabla